# Preprocessamento — FolhaUOL (Colab)

Adaptador fino que monta o Google Drive, atualiza o repositório e chama `scripts/preprocessar.py` (ADR 007 dedup + ADR 004 recortes). **Nenhuma lógica científica vive neste notebook** (CLAUDE.md §5).

Pré-requisitos:
- `articles.csv` no Drive em `MyDrive/ptbr-market-classification/data/raw/` (ou ajustar `CAMINHO_CSV` abaixo).
- Runtime padrão (CPU) é suficiente — esta etapa não usa GPU.

## 1. Parâmetros (editar conforme necessário)

In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification.git'  # substituir pela URL do seu fork
RAMO = 'master'

DIR_REPO = '/content/ptbr-market-classification'
DIR_DRIVE = '/content/drive/MyDrive/ptbr-market-classification'

CAMINHO_CSV = f'{DIR_DRIVE}/data/raw/articles.csv'
DIR_DADOS_PROC = f'{DIR_DRIVE}/data/processado'

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / atualizar repositório

In [ ]:
import os, subprocess

if os.path.exists(DIR_REPO):
    subprocess.run(['git', '-C', DIR_REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'checkout', RAMO], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', RAMO, REPO_URL, DIR_REPO], check=True)

os.chdir(DIR_REPO)
print('cwd =', os.getcwd())

## 4. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 5. Verificar presença do CSV bruto

In [ ]:
from pathlib import Path
caminho = Path(CAMINHO_CSV)
assert caminho.exists(), f'CSV bruto não encontrado em {caminho}'
print(f'CSV: {caminho} ({caminho.stat().st_size / 1024**2:.1f} MB)')

## 6. Executar preprocessamento

Gera `corpus_opcao{7,4,3}.parquet`, `enumeracao_opcao4.json`, `hashes.json` em `DIR_DADOS_PROC`.

In [ ]:
import os
os.environ['CAMINHO_CSV'] = CAMINHO_CSV
os.environ['DIR_DADOS_PROC'] = DIR_DADOS_PROC

!python scripts/preprocessar.py

## 7. Resumo dos artefatos

In [ ]:
import json
from pathlib import Path
import pandas as pd

proc = Path(DIR_DADOS_PROC)
print('Arquivos em', proc)
for p in sorted(proc.glob('*')):
    print(f'  {p.name} ({p.stat().st_size / 1024:.1f} KB)')

print()
print('Hashes:')
print(json.dumps(json.loads((proc / 'hashes.json').read_text()), indent=2))

print()
print('Contagens por recorte:')
for recorte in ('opcao7', 'opcao4', 'opcao3'):
    df = pd.read_parquet(proc / f'corpus_{recorte}.parquet')
    print(f'  {recorte}: {len(df):,} artigos, {df["y_colapsado"].nunique()} classes')